# PIQA — CoT Baseline Evaluation
## Chain-of-Thought vs No-Guidance Baseline

**Purpose:** Runs TWO conditions on the same 300 PIQA questions (seed=42):
- **CoT:** Qwen 1.5B × 5 votes with `Let's think step by step`
- **Baseline:** Qwen 1.5B × 5 votes, no CoT, no guide

Compare against **Guided pipeline** results from the original notebook.

| Condition | Compute | Description |
|---|---|---|
| Baseline | 7.5B | No guide, no CoT |
| **CoT (this notebook)** | **7.5B** | No guide, 'think step by step' |
| Guided (original notebook) | 10.5B | Fine-tuned 3B guide + 1.5B solver |

**Why PIQA?** PIQA tests physical commonsense reasoning — the model must choose the *correct* procedure for a real-world goal from two candidates. This requires reasoning beyond surface pattern matching, making it a meaningful benchmark for CoT vs Baseline.

**Task format:** Given a `goal` and two solutions (`sol1`, `sol2`), the model must output `1` (sol1 is correct) or `2` (sol2 is correct).

**Same seed=42, same 300 questions — direct 3-way comparison is valid.**

In [1]:
# CELL 1 -- Install
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

Done.


In [2]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login

login("")  # paste your token here
print("HuggingFace login done")

HuggingFace login done


In [3]:
# CELL 3 -- Imports + GPU check
import os, json, re, random, time
import torch
import numpy as np
from collections import Counter, defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/piqa_cot_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Output  : {OUTPUT_DIR}")

PyTorch : 2.9.0+cu126
GPU     : Tesla P100-PCIE-16GB
VRAM    : 17.1 GB
Output  : /kaggle/working/piqa_cot_eval


In [6]:
# CELL 4 -- Configuration
# NOTE: No guide model. Only the 1.5B solver.
# CRITICAL: max_eval_samples=300 and random_seed=42 MUST match original notebook.
CONFIG = {
    "solver_model"     : "Qwen/Qwen2.5-1.5B-Instruct",
    "dataset_name"     : "nthngdy/piqa",
    "dataset_split"    : "validation",
    "max_eval_samples" : 500,   # MUST match original notebook
    "random_seed"      : 42,    # MUST match original notebook
    "n_votes"          : 5,
    "vote_temperature" : 0.4,   # MUST match original notebook
    "max_new_tokens"   : 200,   # PIQA answers are short (just reasoning + final choice)
    "solver_params_B"  : 1.5,
    "results_file"     : f"{OUTPUT_DIR}/results.jsonl",
    "angle1_file"      : f"{OUTPUT_DIR}/angle1_compute.json",
    "angle2_file"      : f"{OUTPUT_DIR}/angle2_consistency.json",
    "angle3_file"      : f"{OUTPUT_DIR}/angle3_calibration.json",
    "angle4_file"      : f"{OUTPUT_DIR}/angle4_position_bias.json",
    "checkpoint_file"  : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"       : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")

Config ready:
  solver_model            : Qwen/Qwen2.5-1.5B-Instruct
  dataset_name            : nthngdy/piqa
  dataset_split           : validation
  max_eval_samples        : 500
  random_seed             : 42
  n_votes                 : 5
  vote_temperature        : 0.4
  max_new_tokens          : 200
  solver_params_B         : 1.5
  results_file            : /kaggle/working/piqa_cot_eval/results.jsonl
  angle1_file             : /kaggle/working/piqa_cot_eval/angle1_compute.json
  angle2_file             : /kaggle/working/piqa_cot_eval/angle2_consistency.json
  angle3_file             : /kaggle/working/piqa_cot_eval/angle3_calibration.json
  angle4_file             : /kaggle/working/piqa_cot_eval/angle4_position_bias.json
  checkpoint_file         : /kaggle/working/piqa_cot_eval/checkpoint.json
  save_every              : 25


In [7]:
# CELL 5 -- Load PIQA dataset
# PIQA fields: goal (str), sol1 (str), sol2 (str), label (int: 0=sol1 correct, 1=sol2 correct)
# We map label -> answer string: label=0 -> "1" (sol1), label=1 -> "2" (sol2)
# Same seed=42 and max_eval_samples=300 guarantees the same 300 questions as original.

print("Loading PIQA...")
raw_ds = load_dataset(CONFIG["dataset_name"])
print(f"Splits     : {list(raw_ds.keys())}")
print(f"Val size   : {len(raw_ds[CONFIG['dataset_split']])}")


def normalise_piqa(item):
    """Normalise a PIQA item into the standard evaluation format.

    answer is '1' if label=0 (sol1 is correct) or '2' if label=1 (sol2 is correct).
    answer_label stores the raw 0/1 for position-bias analysis (Angle 4).
    """
    label = int(item["label"])
    ans_str = str(label + 1)   # 0 -> "1", 1 -> "2"
    return {
        "question" : item["goal"].strip(),
        "sol1"     : item["sol1"].strip(),
        "sol2"     : item["sol2"].strip(),
        "answer"   : ans_str,
        "answer_label": label,   # 0 = sol1 correct, 1 = sol2 correct
    }


all_data = [normalise_piqa(x) for x in raw_ds[CONFIG["dataset_split"]]]

# Fix seed ONCE -- must match original notebook
random.seed(CONFIG["random_seed"])
test_data = random.sample(all_data, CONFIG["max_eval_samples"])

label_counts = Counter(d["answer_label"] for d in test_data)
print(f"Sampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
print(f"  sol1 correct (label=0): {label_counts[0]}")
print(f"  sol2 correct (label=1): {label_counts[1]}")
print(f"First goal : {test_data[0]['question'][:80]}")
print(f"  sol1  : {test_data[0]['sol1'][:60]}")
print(f"  sol2  : {test_data[0]['sol2'][:60]}")
print(f"  answer: {test_data[0]['answer']}")
print("PIQA loaded")

Loading PIQA...


README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.66M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/502k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/301k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16113 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3084 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1838 [00:00<?, ? examples/s]

Splits     : ['train', 'test', 'validation']
Val size   : 1838
Sampled 500 questions (seed=42)
  sol1 correct (label=0): 249
  sol2 correct (label=1): 251
First goal : How do I choose good apples at the grocery store?
  sol1  : Look for obvious bad spots. If you see spots that are rotten
  sol2  : Look for obvious bad spots. If you see spots that are rotten
  answer: 1
PIQA loaded


In [8]:
# CELL 6 -- Answer extraction
# PIQA is a binary-choice task: model must output "1" (sol1) or "2" (sol2).
# GT answer is always "1" or "2".

def extract_gt_answer(answer_str):
    """GT answer is already '1' or '2' from normalise_piqa."""
    return str(answer_str).strip()


def extract_pred_answer(text):
    """Extract '1' or '2' from model output.

    Tries patterns in order of precision:
      1. Explicit markers:  #### 1 / #### 2
      2. 'answer is: 1' / 'answer is: 2'
      3. 'solution 1' / 'solution 2'
      4. 'option 1' / 'option 2'
      5. Standalone '1' or '2' at end of text
    Returns '' if none found.
    """
    t = text.strip()

    # #### marker (matches our system prompt convention)
    m = re.search(r"####\s*([12])", t)
    if m: return m.group(1)

    # 'the answer is X' or 'answer: X'
    m = re.search(r"(?:the\s+)?answer\s*(?:is)?\s*:?\s*([12])\b", t, re.IGNORECASE)
    if m: return m.group(1)

    # 'solution 1' / 'solution 2'
    m = re.search(r"solution\s+([12])\b", t, re.IGNORECASE)
    if m: return m.group(1)

    # 'option 1' / 'option 2'
    m = re.search(r"option\s+([12])\b", t, re.IGNORECASE)
    if m: return m.group(1)

    # 'choose 1' / 'choose 2' / 'select 1'
    m = re.search(r"(?:choose|select|pick)\s+([12])\b", t, re.IGNORECASE)
    if m: return m.group(1)

    # **1** or **2** (bold markdown)
    m = re.search(r"\*\*([12])\*\*", t)
    if m: return m.group(1)

    # standalone 1 or 2 at end of line / end of text
    m = re.search(r"\b([12])\s*$", t)
    if m: return m.group(1)

    return ""


# Self-test
_tests = [
    ("#### 1", "1"),
    ("#### 2", "2"),
    ("The answer is 1", "1"),
    ("answer: 2", "2"),
    ("Solution 1 is correct.", "1"),
    ("I would choose solution 2", "2"),
    ("Option 1 makes more sense.", "1"),
    ("**2**", "2"),
    ("The best approach is 2", "2"),
    ("Some irrelevant text here.", ""),
]
all_ok = all(extract_pred_answer(t) == e for t, e in _tests)
print("Extractor:", "ALL PASSED" if all_ok else "FAILURES -- fix before running eval")
if not all_ok:
    for t, e in _tests:
        got = extract_pred_answer(t)
        if got != e:
            print(f"  FAIL: input={repr(t)}  expected={repr(e)}  got={repr(got)}")

Extractor: ALL PASSED


In [9]:
# CELL 7 -- Load solver model (Qwen 1.5B only -- no guide model needed)
# T4 has 15GB. 1.5B in float16 ~ 3GB. Plenty of headroom.

print(f"Loading: {CONFIG['solver_model']}")
solver_tok = AutoTokenizer.from_pretrained(CONFIG["solver_model"])
if solver_tok.pad_token is None:
    solver_tok.pad_token = solver_tok.eos_token

solver_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["solver_model"],
    dtype=torch.float16,
    device_map="auto",
).eval()

vram = torch.cuda.memory_allocated() / 1e9
print(f"VRAM used : {vram:.2f} GB")
print(f"Headroom  : {15.0 - vram:.1f} GB")
print("Solver ready")

Loading: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

VRAM used : 3.09 GB
Headroom  : 11.9 GB
Solver ready


In [10]:
# CELL 8 -- Prompts and generation functions
#
# COT_SYSTEM      -- adds 'Let's think step by step' (the new CoT condition)
# BASELINE_SYSTEM -- no CoT, no guide  (matches original notebook baseline)
# REFINER_SYSTEM  -- tie-breaker (same as original)
#
# PIQA task: given a goal and two solutions, the model must output '1' or '2'.

COT_SYSTEM = (
    "You are a physical commonsense reasoning expert.\n"
    "Let's think step by step.\n"
    "You will be given a goal and two candidate solutions.\n"
    "Reason carefully about which solution correctly achieves the goal.\n"
    "Consider physical plausibility, safety, and practicality.\n"
    "Your FINAL line must be exactly: #### 1  OR  #### 2\n"
    "Do not write anything after the final line.\n\n"
    "Example:\n"
    "Goal: Boil water quickly.\n"
    "Solution 1: Put water in a pot and heat on a stove.\n"
    "Solution 2: Leave water in a cup on the counter.\n"
    "Step 1: Heating on a stove applies direct heat, which boils water.\n"
    "Step 2: Leaving water on the counter does not apply heat.\n"
    "Solution 1 achieves the goal.\n"
    "#### 1"
)

BASELINE_SYSTEM = (
    "You are a physical commonsense reasoning expert.\n"
    "You will be given a goal and two candidate solutions.\n"
    "Choose the solution that correctly achieves the goal.\n"
    "Your FINAL line must be exactly: #### 1  OR  #### 2"
)

REFINER_SYSTEM = (
    "You are a careful physical commonsense reasoning expert.\n"
    "Previous attempts gave different answers. Ignore all of them.\n"
    "Re-evaluate both solutions from scratch.\n"
    "Your FINAL line must be exactly: #### 1  OR  #### 2"
)


def format_piqa_question(goal, sol1, sol2):
    """Format the PIQA question for the model."""
    return (
        f"Goal: {goal}\n"
        f"Solution 1: {sol1}\n"
        f"Solution 2: {sol2}\n"
        "Which solution correctly achieves the goal? Answer with 1 or 2."
    )


def run_solver(messages, temperature):
    prompt = solver_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = solver_tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    device = next(solver_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = solver_model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=max(temperature, 0.05),
            do_sample=True,
            top_p=0.92,
            top_k=40,
            pad_token_id=solver_tok.eos_token_id,
            repetition_penalty=1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return solver_tok.decode(new_toks, skip_special_tokens=True).strip()


def generate_cot(item):
    """CoT: Let's think step by step -- no guide."""
    content = format_piqa_question(item["question"], item["sol1"], item["sol2"])
    return run_solver(
        [{"role": "system", "content": COT_SYSTEM},
         {"role": "user",   "content": content}],
        temperature=CONFIG["vote_temperature"],
    )


def generate_baseline(item):
    """Baseline: no CoT, no guide."""
    content = format_piqa_question(item["question"], item["sol1"], item["sol2"])
    return run_solver(
        [{"role": "system", "content": BASELINE_SYSTEM},
         {"role": "user",   "content": content}],
        temperature=CONFIG["vote_temperature"],
    )


def generate_refiner(item, candidates):
    """Tie-breaker: re-evaluate from scratch ignoring previous votes."""
    cands = ", ".join(sorted(set(c for c in candidates if c)))
    content = (
        f"{format_piqa_question(item['question'], item['sol1'], item['sol2'])}\n\n"
        f"Previous attempts gave: {cands}\n"
        "Ignore all previous attempts. Re-evaluate from scratch:"
    )
    return run_solver(
        [{"role": "system", "content": REFINER_SYSTEM},
         {"role": "user",   "content": content}],
        temperature=0.3,
    )


print("Generation functions ready")
print("  CoT prompt   : 'Let's think step by step'")
print("  Temperature  :", CONFIG["vote_temperature"])

Generation functions ready
  CoT prompt   : 'Let's think step by step'
  Temperature  : 0.4


In [11]:
# CELL 9 -- Voting logic (identical to original notebook, adapted for PIQA item dict)

def vote_and_decide(answers, item, gt_answer=None):
    valid = [a for a in answers if a and a.strip()]
    if not valid: valid = answers

    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(valid)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        ref_raw  = generate_refiner(item, list(valid))
        ref_ans  = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None
        all_v      = valid + ([ref_ans] if ref_ans else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]
        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_v), 4)
        total      = len(all_v)
        correct_votes    = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted           = total - new_top_c
        vote_counts      = new_counts

    return {
        "final_answer": final, "strategy": strategy,
        "confidence": conf, "vote_counts": dict(vote_counts),
        "correct_votes": correct_votes, "total_votes": total,
        "vote_consistency": round(vote_consistency, 4),
        "wasted_votes": wasted,
        "refiner_used": refiner_used, "refiner_correct": refiner_correct,
    }

print("Voting logic ready")

Voting logic ready


In [12]:
# CELL 10 -- Single question test (verify both conditions work before full run)

item = test_data[0]
gt   = extract_gt_answer(item["answer"])

print("=" * 65)
print(f"Goal     : {item['question']}")
print(f"Sol 1    : {item['sol1']}")
print(f"Sol 2    : {item['sol2']}")
print(f"GT Answer: {gt}  (sol{gt} is correct)")

# CoT
print("\n[CoT] 3 sample votes:")
for i in range(3):
    raw  = generate_cot(item)
    pred = extract_pred_answer(raw)
    print(f"  Vote {i+1}: '{pred}'  | raw[:80]: {raw[:80]}")

# Baseline
print("\n[Baseline] 3 sample votes:")
for i in range(3):
    raw  = generate_baseline(item)
    pred = extract_pred_answer(raw)
    print(f"  Vote {i+1}: '{pred}'")

print("\nSingle test done. Run Cell 11 for full evaluation.")

Goal     : How do I choose good apples at the grocery store?
Sol 1    : Look for obvious bad spots. If you see spots that are rotten, dark brown, or too soft, the apple has likely already gone bad. ...    Look for cuts. ...    Examine the color. ...    Check the apple for firmness. ...    Sniff the apple to detect foul odor.
Sol 2    : Look for obvious bad spots. If you see spots that are rotten, dark brown, or too soft, the apple has likely already gone bad. ...    Look for cuts cause those are the good ones. ...    Examine the color. ...    Check the apple for firmness. ...    Sniff the apple to detect foul odor.
GT Answer: 1  (sol1 is correct)

[CoT] 3 sample votes:
  Vote 1: '1'  | raw[:80]: 1
  Vote 2: '1'  | raw[:80]: #### 1
  Vote 3: '1'  | raw[:80]: #### 1

[Baseline] 3 sample votes:
  Vote 1: '1'
  Vote 2: '1'
  Vote 3: '1'

Single test done. Run Cell 11 for full evaluation.


In [13]:
# CELL 11 -- Full Dual Evaluation Loop
# Runs all 300 questions under TWO conditions:
#   Mode A: cot      (1.5B x5, 'Let's think step by step')
#   Mode B: baseline (1.5B x5, no CoT)
# Checkpoints every 25 questions.

print(f"Dual evaluation: {len(test_data)} PIQA questions")
print(f"Each question  : {CONFIG['n_votes']} CoT votes + {CONFIG['n_votes']} baseline votes")
print("-" * 65)

cot_results  = []
base_results = []
start_idx    = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        cot_results  = [r for r in lines if r.get("mode") == "cot"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from {start_idx} (CoT: {len(cot_results)}, Base: {len(base_results)})")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="PIQA CoT Eval"):
    item      = test_data[idx]
    gt_answer = extract_gt_answer(item["answer"])
    ans_label = item["answer_label"]  # 0 or 1 (for position-bias analysis)

    # ---- CoT condition ----------------------------------------
    try:
        cot_votes_raw = [extract_pred_answer(generate_cot(item))
                         for _ in range(CONFIG["n_votes"])]
        c_dec = vote_and_decide(cot_votes_raw, item, gt_answer)
        cot_results.append({
            "mode": "cot", "idx": idx,
            "goal": item["question"],
            "sol1": item["sol1"], "sol2": item["sol2"],
            "gt_answer": gt_answer, "answer_label": ans_label,
            "final_answer": c_dec["final_answer"],
            "correct": c_dec["final_answer"] == gt_answer,
            "strategy": c_dec["strategy"], "confidence": c_dec["confidence"],
            "correct_votes": c_dec["correct_votes"], "total_votes": c_dec["total_votes"],
            "vote_consistency": c_dec["vote_consistency"],
            "wasted_votes": c_dec["wasted_votes"],
            "refiner_used": c_dec["refiner_used"],
            "refiner_correct": c_dec["refiner_correct"],
            "vote_counts": c_dec["vote_counts"],
        })
    except RuntimeError as e:
        cot_results.append({
            "mode": "cot", "idx": idx,
            "goal": item["question"],
            "sol1": item["sol1"], "sol2": item["sol2"],
            "gt_answer": gt_answer, "answer_label": ans_label,
            "final_answer": "", "correct": False, "strategy": "error",
            "confidence": 0.0, "correct_votes": 0,
            "total_votes": CONFIG["n_votes"], "vote_consistency": 0.0,
            "wasted_votes": CONFIG["n_votes"], "refiner_used": False,
            "refiner_correct": None, "vote_counts": {}, "error": str(e),
        })

    # ---- Baseline condition -----------------------------------
    try:
        base_votes_raw = [extract_pred_answer(generate_baseline(item))
                          for _ in range(CONFIG["n_votes"])]
        b_dec = vote_and_decide(base_votes_raw, item, gt_answer)
        base_results.append({
            "mode": "baseline", "idx": idx,
            "goal": item["question"],
            "sol1": item["sol1"], "sol2": item["sol2"],
            "gt_answer": gt_answer, "answer_label": ans_label,
            "final_answer": b_dec["final_answer"],
            "correct": b_dec["final_answer"] == gt_answer,
            "strategy": b_dec["strategy"], "confidence": b_dec["confidence"],
            "correct_votes": b_dec["correct_votes"], "total_votes": b_dec["total_votes"],
            "vote_consistency": b_dec["vote_consistency"],
            "wasted_votes": b_dec["wasted_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": b_dec["vote_counts"],
        })
    except RuntimeError as e:
        base_results.append({
            "mode": "baseline", "idx": idx,
            "goal": item["question"],
            "sol1": item["sol1"], "sol2": item["sol2"],
            "gt_answer": gt_answer, "answer_label": ans_label,
            "final_answer": "", "correct": False, "strategy": "error",
            "confidence": 0.0, "correct_votes": 0,
            "total_votes": CONFIG["n_votes"], "vote_consistency": 0.0,
            "wasted_votes": CONFIG["n_votes"], "refiner_used": False,
            "refiner_correct": None, "vote_counts": {}, "error": str(e),
        })

    # Checkpoint
    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in cot_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        c_acc = sum(r["correct"] for r in cot_results)  / len(cot_results)  * 100
        b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
        mins  = (time.time() - t0) / 60
        print(f"  [{idx+1:3d}] CoT: {c_acc:.1f}%  Baseline: {b_acc:.1f}%  ({mins:.1f} min)")

# Final save
with open(CONFIG["results_file"], "w") as f:
    for r in cot_results + base_results:
        f.write(json.dumps(r) + "\n")

c_c = sum(r["correct"] for r in cot_results)
b_c = sum(r["correct"] for r in base_results)
print(f"\nDone. CoT: {c_c}/{len(cot_results)} = {c_c/len(cot_results)*100:.1f}%")
print(f"Baseline : {b_c}/{len(base_results)} = {b_c/len(base_results)*100:.1f}%")
print(f"CoT vs Baseline: {(c_c/len(cot_results) - b_c/len(base_results))*100:+.1f} pts")

Dual evaluation: 500 PIQA questions
Each question  : 5 CoT votes + 5 baseline votes
-----------------------------------------------------------------
Starting fresh


PIQA CoT Eval:   0%|          | 0/500 [00:00<?, ?it/s]

  [ 25] CoT: 60.0%  Baseline: 48.0%  (0.7 min)
  [ 50] CoT: 50.0%  Baseline: 52.0%  (1.3 min)
  [ 75] CoT: 46.7%  Baseline: 52.0%  (1.9 min)
  [100] CoT: 47.0%  Baseline: 52.0%  (2.6 min)
  [125] CoT: 46.4%  Baseline: 54.4%  (3.2 min)
  [150] CoT: 48.7%  Baseline: 53.3%  (3.7 min)
  [175] CoT: 50.3%  Baseline: 55.4%  (4.3 min)
  [200] CoT: 50.5%  Baseline: 54.0%  (5.0 min)
  [225] CoT: 51.1%  Baseline: 54.7%  (5.6 min)
  [250] CoT: 50.4%  Baseline: 54.8%  (6.2 min)
  [275] CoT: 50.9%  Baseline: 55.6%  (6.7 min)
  [300] CoT: 52.0%  Baseline: 56.0%  (7.3 min)
  [325] CoT: 52.9%  Baseline: 56.0%  (8.0 min)
  [350] CoT: 53.7%  Baseline: 54.9%  (8.6 min)
  [375] CoT: 53.1%  Baseline: 55.2%  (9.3 min)
  [400] CoT: 53.5%  Baseline: 54.8%  (9.9 min)
  [425] CoT: 54.8%  Baseline: 55.8%  (10.5 min)
  [450] CoT: 54.7%  Baseline: 55.6%  (11.2 min)
  [475] CoT: 54.1%  Baseline: 55.4%  (11.8 min)
  [500] CoT: 54.8%  Baseline: 55.4%  (12.4 min)

Done. CoT: 274/500 = 54.8%
Baseline : 277/500 = 55.4%
C

In [14]:
# CELL 12 -- ANGLE 1: COMPUTE EFFICIENCY
# Both CoT and Baseline cost 7.5B (1.5B x5). Zero guide overhead.

S, N = CONFIG["solver_params_B"], CONFIG["n_votes"]
compute = S * N  # 7.5B for both

c_acc = sum(r["correct"] for r in cot_results)  / len(cot_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
c_wasted = sum(r["wasted_votes"] for r in cot_results)
b_wasted = sum(r["wasted_votes"] for r in base_results)
c_ref = sum(r["refiner_used"] for r in cot_results)
b_ref = sum(r["refiner_used"] for r in base_results)

# Extraction failure rate (model outputs neither '1' nor '2')
c_fail = sum(1 for r in cot_results  if r["final_answer"] not in ("1", "2"))
b_fail = sum(1 for r in base_results if r["final_answer"] not in ("1", "2"))

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (PIQA)")
print("=" * 65)
print(f"\n  {'Setup':<28} | {'Compute':>8} | {'Accuracy':>9}")
print(f"  {'-'*28}-+-{'-'*8}-+-{'-'*9}")
print(f"  {'Baseline (no CoT)':<28} | {compute:>6.1f}B  | {b_acc:>8.1f}%")
print(f"  {'CoT (think step by step)':<28} | {compute:>6.1f}B  | {c_acc:>8.1f}%")
print(f"\n  Accuracy gain (CoT vs Baseline): {c_acc - b_acc:+.1f} pts")
print(f"  Compute difference              : 0 (same 7.5B)")
print(f"  Wasted votes saved              : {b_wasted - c_wasted} ({c_wasted} CoT vs {b_wasted} Baseline)")
print(f"  Refiner: CoT={c_ref}  Baseline={b_ref}")
print(f"  Extraction failures: CoT={c_fail}  Baseline={b_fail}")

angle1 = {
    "dataset": "PIQA", "n_questions": len(cot_results), "compute_B": compute,
    "cot_accuracy": round(c_acc, 2), "baseline_accuracy": round(b_acc, 2),
    "accuracy_gain": round(c_acc - b_acc, 2),
    "cot_wasted": c_wasted, "baseline_wasted": b_wasted,
    "cot_refiner": c_ref, "baseline_refiner": b_ref,
    "cot_extraction_failures": c_fail, "baseline_extraction_failures": b_fail,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(angle1, f, indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")

ANGLE 1 -- COMPUTE EFFICIENCY  (PIQA)

  Setup                        |  Compute |  Accuracy
  -----------------------------+----------+----------
  Baseline (no CoT)            |    7.5B  |     55.4%
  CoT (think step by step)     |    7.5B  |     54.8%

  Accuracy gain (CoT vs Baseline): -0.6 pts
  Compute difference              : 0 (same 7.5B)
  Wasted votes saved              : -277 (347 CoT vs 70 Baseline)
  Refiner: CoT=0  Baseline=0
  Extraction failures: CoT=0  Baseline=0

Saved -> /kaggle/working/piqa_cot_eval/angle1_compute.json


In [15]:
# CELL 13 -- ANGLE 2: VOTE CONSISTENCY

c_cons = [r["vote_consistency"] for r in cot_results]
b_cons = [r["vote_consistency"] for r in base_results]
c_mean = np.mean(c_cons)
b_mean = np.mean(b_cons)
lift   = c_mean / max(b_mean, 1e-6)

cot_wins      = sum(1 for c, b in zip(c_cons, b_cons) if c > b)
baseline_wins = sum(1 for c, b in zip(c_cons, b_cons) if b > c)
tied          = sum(1 for c, b in zip(c_cons, b_cons) if c == b)

def bucket(scores):
    return {
        "all_wrong (0%)":    sum(1 for s in scores if s == 0.0),
        "low (1-39%)":       sum(1 for s in scores if 0.0 < s < 0.4),
        "medium (40-79%)":   sum(1 for s in scores if 0.4 <= s < 0.8),
        "high (80-100%)":    sum(1 for s in scores if s >= 0.8),
    }

c_dist = bucket(c_cons)
b_dist = bucket(b_cons)

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (PIQA)")
print("=" * 65)
print(f"\n  CoT mean consistency     : {c_mean*100:.1f}%")
print(f"  Baseline mean consistency: {b_mean*100:.1f}%")
print(f"  Consistency lift         : {lift:.2f}x")
print(f"\n  CoT wins / Baseline wins / Tied: {cot_wins} / {baseline_wins} / {tied}")
print(f"\n  {'Bucket':<22} | {'CoT':>8} | {'Baseline':>8} | {'Diff':>6}")
for bkt in ["all_wrong (0%)", "low (1-39%)", "medium (40-79%)", "high (80-100%)"]:
    cv, bv = c_dist[bkt], b_dist[bkt]
    print(f"  {bkt:<22} | {cv:>8} | {bv:>8} | {cv-bv:>+6}")

angle2 = {
    "cot_mean_consistency": round(c_mean, 4),
    "baseline_mean_consistency": round(b_mean, 4),
    "consistency_lift": round(lift, 4),
    "cot_wins": cot_wins, "baseline_wins": baseline_wins, "tied": tied,
    "cot_distribution": c_dist, "baseline_distribution": b_dist,
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(angle2, f, indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")

ANGLE 2 -- VOTE CONSISTENCY  (PIQA)

  CoT mean consistency     : 55.8%
  Baseline mean consistency: 55.1%
  Consistency lift         : 1.01x

  CoT wins / Baseline wins / Tied: 183 / 183 / 134

  Bucket                 |      CoT | Baseline |   Diff
  all_wrong (0%)         |      108 |      201 |    -93
  low (1-39%)            |       50 |       13 |    +37
  medium (40-79%)        |      120 |       22 |    +98
  high (80-100%)         |      222 |      264 |    -42

Saved -> /kaggle/working/piqa_cot_eval/angle2_consistency.json


In [16]:
# CELL 14 -- ANGLE 3: CONFIDENCE CALIBRATION

def calibration_report(results, label):
    buckets = [
        ("Very High (>=0.80)",    lambda c: c >= 0.80, 0.90),
        ("High (0.60-0.80)",      lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium (0.40-0.60)",    lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low (<0.40)",           lambda c: c < 0.40, 0.25),
    ]
    n_total    = len(results)
    ece        = 0.0
    calib_out  = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])
    print(f"\n  [{label}]")
    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset: continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"    {name:<22}: n={n}  acc={acc*100:.1f}%  expected={mid*100:.0f}%  gap={gap:.3f}  {flag}")
        calib_out.append({"bucket": name, "count": n, "accuracy": round(acc,4),
                          "expected": mid, "gap": round(gap,4)})
    hc     = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    print(f"    ECE={ece:.4f}  |  High-conf: {len(hc)}  |  Acc@high={hc_acc:.1f}%  |  False-conf: {false_conf}")
    return ece, calib_out, false_conf

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (PIQA)")
print("=" * 65)
c_ece, c_calib, c_false = calibration_report(cot_results,  "CoT")
b_ece, b_calib, b_false = calibration_report(base_results, "Baseline")

improve = (b_ece - c_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE: CoT={c_ece:.4f}  Baseline={b_ece:.4f}  Improvement={improve:.1f}%")
print(f"  False confidence: CoT={c_false}  Baseline={b_false}")

angle3 = {
    "cot_ece": round(c_ece, 4), "baseline_ece": round(b_ece, 4),
    "ece_improvement_pct": round(improve, 2),
    "cot_false_conf": c_false, "baseline_false_conf": b_false,
    "false_conf_reduction": b_false - c_false,
    "cot_calibration": c_calib, "baseline_calibration": b_calib,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(angle3, f, indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")

ANGLE 3 -- CONFIDENCE CALIBRATION  (PIQA)

  [CoT]
    Very High (>=0.80)    : n=380  acc=58.4%  expected=90%  gap=0.316  Poor
    High (0.60-0.80)      : n=120  acc=43.3%  expected=70%  gap=0.267  Poor
    ECE=0.3040  |  High-conf: 380  |  Acc@high=58.4%  |  False-conf: 158

  [Baseline]
    Very High (>=0.80)    : n=478  acc=55.2%  expected=90%  gap=0.348  Poor
    High (0.60-0.80)      : n=22  acc=59.1%  expected=70%  gap=0.109  Good
    ECE=0.3372  |  High-conf: 478  |  Acc@high=55.2%  |  False-conf: 214

  ECE: CoT=0.3040  Baseline=0.3372  Improvement=9.8%
  False confidence: CoT=158  Baseline=214

Saved -> /kaggle/working/piqa_cot_eval/angle3_calibration.json


In [17]:
# CELL 15 -- ANGLE 4: POSITION BIAS ANALYSIS
# PIQA-specific: does the model prefer sol1 or sol2 regardless of correctness?
#
# answer_label=0 -> correct answer is sol1 ("1")
# answer_label=1 -> correct answer is sol2 ("2")
#
# Position bias = tendency to always pick "1" or always pick "2".
# If CoT reduces position bias, that shows genuine reasoning over anchoring.

def position_bias_report(results, label):
    # Accuracy when correct answer is sol1 vs sol2
    sol1_correct_items = [r for r in results if r["answer_label"] == 0]
    sol2_correct_items = [r for r in results if r["answer_label"] == 1]

    sol1_acc = sum(r["correct"] for r in sol1_correct_items) / max(len(sol1_correct_items), 1) * 100
    sol2_acc = sum(r["correct"] for r in sol2_correct_items) / max(len(sol2_correct_items), 1) * 100

    # Prediction distribution: how often model picks "1" vs "2"
    pred_counts = Counter(r["final_answer"] for r in results)
    n = len(results)
    pct_pick1 = pred_counts.get("1", 0) / n * 100
    pct_pick2 = pred_counts.get("2", 0) / n * 100
    pct_fail  = (n - pred_counts.get("1", 0) - pred_counts.get("2", 0)) / n * 100

    # Position bias score: |P(pick=1) - 0.5| -- 0 = no bias, 0.5 = always picks same
    bias_score = abs(pct_pick1 / 100 - 0.5)

    print(f"\n  [{label}]")
    print(f"    Acc when sol1 is correct (n={len(sol1_correct_items)}): {sol1_acc:.1f}%")
    print(f"    Acc when sol2 is correct (n={len(sol2_correct_items)}): {sol2_acc:.1f}%")
    print(f"    Acc gap (sol1 - sol2)    : {sol1_acc - sol2_acc:+.1f}%")
    print(f"    Prediction dist: pick_1={pct_pick1:.1f}%  pick_2={pct_pick2:.1f}%  fail={pct_fail:.1f}%")
    print(f"    Position bias score      : {bias_score:.3f}  (0=unbiased, 0.5=fully biased)")

    return {
        "n_sol1_correct": len(sol1_correct_items),
        "n_sol2_correct": len(sol2_correct_items),
        "acc_when_sol1_correct": round(sol1_acc, 2),
        "acc_when_sol2_correct": round(sol2_acc, 2),
        "acc_gap": round(sol1_acc - sol2_acc, 2),
        "pct_predict_1": round(pct_pick1, 2),
        "pct_predict_2": round(pct_pick2, 2),
        "pct_extraction_fail": round(pct_fail, 2),
        "position_bias_score": round(bias_score, 4),
    }


print("=" * 65)
print("ANGLE 4 -- POSITION BIAS ANALYSIS  (PIQA)")
print("=" * 65)
c_bias = position_bias_report(cot_results,  "CoT")
b_bias = position_bias_report(base_results, "Baseline")

bias_reduction = b_bias["position_bias_score"] - c_bias["position_bias_score"]
print(f"\n  Bias reduction from CoT: {bias_reduction:+.4f}")
if bias_reduction > 0.02:
    print("  RESULT: CoT reduces position bias -- model reasons more genuinely.")
elif abs(bias_reduction) <= 0.02:
    print("  RESULT: Minimal bias change -- CoT does not reduce anchoring.")
else:
    print("  RESULT: CoT increases position bias -- unexpected, worth investigating.")

angle4 = {
    "dataset": "PIQA",
    "cot": c_bias,
    "baseline": b_bias,
    "bias_reduction": round(bias_reduction, 4),
}
with open(CONFIG["angle4_file"], "w") as f:
    json.dump(angle4, f, indent=2)
print(f"\nSaved -> {CONFIG['angle4_file']}")

ANGLE 4 -- POSITION BIAS ANALYSIS  (PIQA)

  [CoT]
    Acc when sol1 is correct (n=249): 49.0%
    Acc when sol2 is correct (n=251): 60.6%
    Acc gap (sol1 - sol2)    : -11.6%
    Prediction dist: pick_1=44.2%  pick_2=55.8%  fail=0.0%
    Position bias score      : 0.058  (0=unbiased, 0.5=fully biased)

  [Baseline]
    Acc when sol1 is correct (n=249): 98.0%
    Acc when sol2 is correct (n=251): 13.1%
    Acc gap (sol1 - sol2)    : +84.8%
    Prediction dist: pick_1=92.4%  pick_2=7.6%  fail=0.0%
    Position bias score      : 0.424  (0=unbiased, 0.5=fully biased)

  Bias reduction from CoT: +0.3660
  RESULT: CoT reduces position bias -- model reasons more genuinely.

Saved -> /kaggle/working/piqa_cot_eval/angle4_position_bias.json


In [18]:
# CELL 16 -- 3-WAY COMPARISON SUMMARY (Paper Table)
# Paste your Guided results from the original notebook below.

# ---- Paste guided results from original notebook ----
GUIDED_ACC     = 78.0   # guided accuracy %
GUIDED_ECE     = 0.100  # guided ECE
GUIDED_COMPUTE = 10.5   # B param-passes
# -----------------------------------------------------

c_acc = sum(r["correct"] for r in cot_results)  / len(cot_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
c_ece = json.load(open(CONFIG["angle3_file"]))["cot_ece"]
b_ece = json.load(open(CONFIG["angle3_file"]))["baseline_ece"]

c_bias_score = json.load(open(CONFIG["angle4_file"]))["cot"]["position_bias_score"]
b_bias_score = json.load(open(CONFIG["angle4_file"]))["baseline"]["position_bias_score"]

print("=" * 72)
print("  PIQA -- 3-WAY COMPARISON  (Paper Table)")
print("=" * 72)
print(f"  N={len(cot_results)}  Seed={CONFIG['random_seed']}  Solver=Qwen2.5-1.5B")
print()
print(f"  {'Condition':<30} | {'Compute':>7} | {'Accuracy':>9} | {'ECE':>7} | {'vs Baseline':>12}")
print(f"  {'-'*30}-+-{'-'*7}-+-{'-'*9}-+-{'-'*7}-+-{'-'*12}")
print(f"  {'Baseline (no guide, no CoT)':<30} | {'7.5B':>7} | {b_acc:>8.1f}% | {b_ece:>7.4f} | {'—':>12}")
print(f"  {'CoT (think step by step)':<30} | {'7.5B':>7} | {c_acc:>8.1f}% | {c_ece:>7.4f} | {c_acc-b_acc:>+11.1f}%")
print(f"  {'Guided (fine-tuned 3B guide)':<30} | {'10.5B':>7} | {GUIDED_ACC:>8.1f}% | {GUIDED_ECE:>7.4f} | {GUIDED_ACC-b_acc:>+11.1f}%")
print()
print(f"  POSITION BIAS: CoT={c_bias_score:.3f}  Baseline={b_bias_score:.3f}")
print()
print(f"  KEY QUESTION: Does Guided beat CoT?")
print(f"  Guided vs CoT: {GUIDED_ACC - c_acc:+.1f} pts")
if GUIDED_ACC > c_acc + 2:
    print("  RESULT: Guided clearly outperforms CoT. Guide model justified.")
elif abs(GUIDED_ACC - c_acc) <= 2:
    print("  RESULT: Marginal difference (<2pts). Report honestly -- discuss tradeoff.")
else:
    print("  RESULT: CoT matches/beats Guided. Important finding to report honestly.")

  PIQA -- 3-WAY COMPARISON  (Paper Table)
  N=500  Seed=42  Solver=Qwen2.5-1.5B

  Condition                      | Compute |  Accuracy |     ECE |  vs Baseline
  -------------------------------+---------+-----------+---------+-------------
  Baseline (no guide, no CoT)    |    7.5B |     55.4% |  0.3372 |            —
  CoT (think step by step)       |    7.5B |     54.8% |  0.3040 |        -0.6%
  Guided (fine-tuned 3B guide)   |   10.5B |     78.0% |  0.1000 |       +22.6%

  POSITION BIAS: CoT=0.058  Baseline=0.424

  KEY QUESTION: Does Guided beat CoT?
  Guided vs CoT: +23.2 pts
  RESULT: Guided clearly outperforms CoT. Guide model justified.
